# Exact Significance Test for Term Dispersion Toy Example

Author: Paul Sheridan

Description: Generate estimated p-values and RICF scores reported in Table 4.

In [8]:
# Imports
import numpy as np
import pandas as pd
import sys
sys.path.append('../')
import wordstats

# Initialize a random seed to ensure results can be replicated
#np.random.seed(998510)
rng = np.random.default_rng(998510)

## Initial Setup

Hardcode the true underlying document collection.

In [9]:
d = 8  # Number of docs
m = 6  # Vocab size

# Terms are rows, docs are columns
nij = np.zeros((m, d), dtype=int)
nij[0, :] = [3, 0, 2, 0, 2, 0, 0, 0]
nij[1, :] = [0, 0, 4, 0, 0, 0, 0, 0]
nij[2, :] = [5, 3, 1, 4, 0, 6, 0, 4]
nij[3, :] = [2, 4, 3, 2, 2, 4, 4, 1]
nij[4, :] = [0, 0, 0, 0, 3, 0, 2, 0]
nij[5, :] = [0, 3, 0, 4, 3, 0, 4, 5]

term_names = [f"t{i}" for i in range(1, m + 1)]
doc_names = [f"d{j}" for j in range(1, d + 1)]

nij_df = pd.DataFrame(nij, index=term_names, columns=doc_names)

print("Term-document matrix:\n")
display(nij_df)

# Term statistics (mirror the R code)
ri = nij.sum(axis=1)                 # total counts per term
nj = nij.sum(axis=0)                 # doc lengths
bij = (nij > 0).astype(int)          # binary presence/absence
bi = bij.sum(axis=1)                 # document frequencies per term
n = int(nj.sum())                    # total terms

print("\nDoc lengths (nj):\n")
display(pd.Series(nj, index=doc_names, name="nj"))

print("\nTotal term counts (ri):\n")
display(pd.Series(ri, index=term_names, name="ri"))

print("\nDocument frequencies (bi):\n")
display(pd.Series(bi, index=term_names, name="bi"))

print("\nTotal terms (n):\n")
print(n)


Term-document matrix:



,d1,d2,d3,d4,d5,d6,d7,d8
t1,3,0,2,0,2,0,0,0
t2,0,0,4,0,0,0,0,0
t3,5,3,1,4,0,6,0,4
t4,2,4,3,2,2,4,4,1
t5,0,0,0,0,3,0,2,0
t6,0,3,0,4,3,0,4,5



Doc lengths (nj):



d1    10
d2    10
d3    10
d4    10
d5    10
d6    10
d7    10
d8    10
Name: nj, dtype: int64


Total term counts (ri):



t1     7
t2     4
t3    23
t4    22
t5     5
t6    19
Name: ri, dtype: int64


Document frequencies (bi):



t1    3
t2    1
t3    6
t4    8
t5    2
t6    5
Name: bi, dtype: int64


Total terms (n):

80


## P-value Estimation

Estimate p-values (Monte Carlo) using the same logic as the R notebook.

In [10]:
R_sat = 100_000  # Number of accepted simulations (per term)
thetas = ri / n
probs_est = np.zeros(m, dtype=float)

for i in range(m):
    R_numer = 0  # numerator
    R_denom = 0  # denominator
    ri_obs = int(ri[i])

    while R_denom < R_sat:
        nij_sim = np.zeros((m, d), dtype=int)

        # Simulate each document column as multinomial with probs=thetas
        # NOTE: Preserves the R code's size = nj[j] - 1
        for j in range(d):
            nij_sim[:, j] = rng.multinomial(int(nj[j] - 1), thetas)

        nj_sim = nij_sim.sum(axis=0)

        # If empty doc exists, discard
        if not np.any(nj_sim == 0):
            row = nij_sim[i, :]
            ri_sim = int(row.sum())
            bi_sim = int((row > 0).sum())

            if bi[i] == bi_sim:
                R_denom += 1

            if (ri_sim >= ri_obs) and (bi[i] == bi_sim):
                R_numer += 1

    probs_est[i] = R_numer / R_denom

    print(f"\nTerm: {i+1}")
    print(f"Numerator: {R_numer}")
    print(f"Denominator: {R_denom}")
    print(f"Estimated probability: {probs_est[i]}")
    print(f"Negative log10 probability: {-np.log10(probs_est[i])}\n")



Term: 1
Numerator: 3642
Denominator: 100000
Estimated probability: 0.03642
Negative log10 probability: 1.4386600585410987


Term: 2
Numerator: 169
Denominator: 100000
Estimated probability: 0.00169
Negative log10 probability: 2.7721132953863266


Term: 3
Numerator: 2602
Denominator: 100000
Estimated probability: 0.02602
Negative log10 probability: 1.5846927077744326


Term: 4
Numerator: 42589
Denominator: 100000
Estimated probability: 0.42589
Negative log10 probability: 0.37070255714579875


Term: 5
Numerator: 2040
Denominator: 100000
Estimated probability: 0.0204
Negative log10 probability: 1.6903698325741012


Term: 6
Numerator: 786
Denominator: 100000
Estimated probability: 0.00786
Negative log10 probability: 2.104577453960592



Calculate RICF scores

In [43]:
import wordstats as ws

# Convert to format accepted by processing functions
ri_mat = np.matrix(ri.reshape(1, m))
nj_mat = np.matrix(nj.reshape(d, 1))
bi_mat = np.matrix(bi.reshape(1, m))

#thetas = ri / n
#print(n)
thetas = np.array(range(1, max(ri_mat.A[0]) + 1))/n
#thetas = np.linspace(0.00001, 0.7, num=100)
print(thetas)


x = ws.eidf_idf_diff(thetas[0], d, nj, bi[0])
print(x)

opt_thetas = ws.get_opt_thetas(n, m, d, ri_mat, nj_mat, bi_mat, thetas)

CF = ws.get_CF(ri_mat)
ICF = ws.get_ICF(CF)

RICF = ws.get_RICF(thetas, n, ICF)

print(RICF)

[0.0125 0.025  0.0375 0.05   0.0625 0.075  0.0875 0.1    0.1125 0.125
 0.1375 0.15   0.1625 0.175  0.1875 0.2    0.2125 0.225  0.2375 0.25
 0.2625 0.275  0.2875]
1.6208376350013300742


ValueError: f(a) and f(b) must have different signs

In [28]:
# Convert to format accepted by processing functions
ri_mat = np.matrix(ri.reshape(1, m))
nj_mat = np.matrix(nj.reshape(d, 1))
bi_mat = np.matrix(bi.reshape(1, m))

print(type(ri_mat.A[0]))

# Compute theta values
thetas = np.array(range(1, max(ri_mat.A[0]) + 1))/n

print(thetas)

opt_thetas = wordstats.get_opt_thetas(n, m, d, ri_mat, nj_mat, bi_mat, thetas)

CF = get_CF(ri_mat)
ICF = get_ICF(CF)

# Compute RICF scores
RICF = wordstats.get_RICF(opt_thetas, n, ICF)

<class 'numpy.ndarray'>
[0.0125 0.025  0.0375 0.05   0.0625 0.075  0.0875 0.1    0.1125 0.125
 0.1375 0.15   0.1625 0.175  0.1875 0.2    0.2125 0.225  0.2375 0.25
 0.2625 0.275  0.2875]


ValueError: f(a) and f(b) must have different signs

In [22]:
print(type(ri))
print(f"Shape of ri: {ri.shape}")
print(ri)
x = ri.reshape(6,1)
x
print(type(x))
print(f"Shape of x: {x.shape}")

y = np.matrix(ri.reshape(1,m))
print(type(y))
print(f"Shape of y: {y.shape}")
print(y)

<class 'numpy.ndarray'>
Shape of ri: (6,)
[ 7  4 23 22  5 19]
<class 'numpy.ndarray'>
Shape of x: (6, 1)
<class 'numpy.matrix'>
Shape of y: (1, 6)
[[ 7  4 23 22  5 19]]


In [ ]:
ri = nij.sum(axis=1)                 # total counts per term
nj = nij.sum(axis=0)                 # doc lengths
bij = (nij > 0).astype(int)          # binary presence/absence
bi = bij.sum(axis=1)                 # document frequencies per term
n = int(nj.sum())                    # total terms

thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)